# 🚲 Bike Demand Forecast
**Question:** How accurately can we estimate next-hour rental demand?

Steps: load → validate → explore → baseline → model → evaluate → limitations.

In [ ]:
# Choose the data mode: "sample" runs offline on the synthetic fixture,
# "real" downloads/queries the official source the first time and caches it in data/raw/.
MODE = "sample"
OUT = f"outputs/notebook-{MODE}"

import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import load_data, analysis

Path(OUT).mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)

> **Sample mode** uses invented data with the real schema. Every chart is stamped *SYNTHETIC SAMPLE*. Change `MODE = "real"` above to run on the official data.

## 1 · Load and validate
`load_data.load()` checks every column, parses dates, and verifies that `casual + registered == cnt` on every row.

In [ ]:
raw = load_data.load(MODE)
print(len(raw), 'rows', raw['timestamp'].min(), '->', raw['timestamp'].max())
raw.head()

## 2 · Explore
Working days show two commuter peaks; weekends one afternoon hump.

In [ ]:
display(Image(f"{OUT}/{analysis.plot_hourly_profile(raw, Path(OUT), MODE == 'sample')}"))

## 3 · The trap: leakage
`casual` and `registered` add up to the target. Watch what happens if we let a model see them:

In [ ]:
from sklearn.linear_model import LinearRegression
leaky = LinearRegression().fit(raw[['casual', 'registered']], raw['cnt'])
print('R² with leaky features:', round(leaky.score(raw[['casual', 'registered']], raw['cnt']), 4), '← perfect, and useless')

## 4 · Decide what is known at prediction time
We predict hour *t* at the end of hour *t−1*: calendar for *t*, demand up to *t−1*, and weather **observed** in *t−1* (the file has observations, not forecasts). Lags are built on a complete hourly grid because the file skips hours.

In [ ]:
frame = analysis.build_features(raw)
print('features:', analysis.FEATURES)
frame[['timestamp', 'cnt', 'cnt_lag1', 'cnt_lag24', 'temp_lag1']].head(30).tail(5)

## 5 · Chronological split and baseline
Random splits leak the future. We hold out the most recent 20 % of the timeline.

In [ ]:
import numpy as np
train, test, cutoff = analysis.chronological_split(frame, 0.2)
print('train <', cutoff.date(), '<= test |', len(train), len(test))
base = analysis.historical_baseline(train, test)
print('Baseline MAE:', round(analysis.mae(test['cnt'], base), 2))

## 6 · Model

In [ ]:
model = analysis.fit_model(train)
pred = np.clip(model.predict(analysis.feature_matrix(test)), 0, None)
print('Model MAE:', round(analysis.mae(test['cnt'], pred), 2))

## 7 · Evaluate

In [ ]:
errs = test.assign(baseline_err=(test['cnt'] - base).abs(), model_err=(test['cnt'] - pred).abs())
by_hour = errs.groupby('hr').agg(baseline_mae=('baseline_err', 'mean'), model_mae=('model_err', 'mean'))
display(Image(f"{OUT}/{analysis.plot_holdout(test, base, pred, Path(OUT), MODE == 'sample')}"))
display(Image(f"{OUT}/{analysis.plot_error_by_hour(by_hour, Path(OUT), MODE == 'sample')}"))

## 8 · Limitations
- Two years, one city; 2012 demand is much higher than 2011.
- Weather is lagged observation, not a forecast — a real system would use forecasts.
- MAE treats a miss of 50 at 3 a.m. the same as at 5 p.m.

## Next
Try the challenges in `README.md`: a second model, a 24-hour horizon, or prediction intervals.